In [ ]:
#---------------------------------------------------------------------
# EXPERIMENT 1: Full-Deck action space vs Hand-Bitmap action space
#---------------------------------------------------------------------
from DQNPlayer_HandBitmap import DQNPlayer_HandBitmap
from Deck import Deck
from Game import GameEnv as Game
from RandomPlayer import RandomPlayer
from DQNPlayer import DQNPlayer
from collections import OrderedDict
from bitarray import bitarray
from Player import Player
import itertools

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

MAX_PLAYS = 100

In [ ]:
################### UTILITIES ######################

from Deck import DeckHelper

cumulated_rewards_learning_agent = []  # Initialize cumulative rewards for the learning agent

def print_game_state(game: Game, players_action: OrderedDict[Player, bitarray], played_cards: bitarray, rewards: dict[Player, int]):
    TOTAL_CARDS = game.deck.num_cards  # Assuming the deck has a num_cards attribute
    print(f"Current Game State:")
    for player in players_action.keys():
        print(f"Player {player.get_id()} - \t Cards Played: {show_bitarray_in_group(game, players_action[player], TOTAL_CARDS//4)}, Num Cards Played: {sum(players_action[player])}, Score: {rewards[player]}")
    print(f"Played Cards: \t      {played_cards}")
    print("-" * 50)

def show_bitarray_in_group(game: Game, bitarr: bitarray, group_size: int) -> str:
    """Helper function to display a bitarray in groups of a specified size for better readability."""
    groups = []
    
    # Iteriamo sui blocchi (es. di 13 in 13 per i semi)
    for start_idx in range(0, len(bitarr), group_size):
        group_bits = bitarr[start_idx : start_idx + group_size]
        cards_str = []
        
        # Iteriamo all'interno del singolo blocco
        for offset, bit in enumerate(group_bits):
            if bit == 1:
                # Calcoliamo l'indice assoluto della carta nel mazzo
                card_idx = start_idx + offset
                rank = DeckHelper.card_to_rank(game.deck, card_idx)
                cards_str.append(str(rank))
            else:
                cards_str.append('0')
                
        groups.append(' '.join(cards_str))
        
    return '    '.join(groups)


def repeat_and_average(n_runs: int = 5):
    """
    Decorator to repeat a function multiple times and average its results.
    """
    def decorator(func):
        def wrapper(*args, **kwargs):
            all_rewards = []
            all_actions_runs: OrderedDict[int, list[list[bitarray]]] = OrderedDict() # Raccoglie la cronologia azioni intera di ogni singola run
            # OrderedDict[int, list[list[bitarray]]]]
            for run in range(n_runs):
                print(f"\n{'='*50}\n INIZIO ESECUZIONE SPERIMENTALE RUN {run + 1}/{n_runs}\n{'='*50}")
                
                rewards, actions = func(*args, **kwargs)
                all_rewards.append(rewards)
                all_actions_runs = actions  # Salva la cronologia delle azioni per questa run

            print(f"\n{'='*50}\n CALCOLO DELLA MEDIA DEI RISULTATI SU {n_runs} RUN...\n{'='*50}")

            averaged_rewards = OrderedDict()
            all_actions = OrderedDict()
            players = list(all_rewards[0].keys())
            player_round_action_list : list[list[bitarray]] = []

            for player in players:
                # --- REWARDS ---
                player_reward_lists = [run_rewards[player] for run_rewards in all_rewards]
                padded_rewards = list(itertools.zip_longest(*player_reward_lists, fillvalue=np.nan))
                
                # nanmean to ignore NaN values when calculating the mean
                mean_rewards = np.nanmean(padded_rewards, axis=1).tolist()
                averaged_rewards[player] = mean_rewards

            

            return averaged_rewards, all_actions_runs
        
        # Preserve the original function's metadata
        wrapper.__name__ = func.__name__
        wrapper.__doc__ = func.__doc__
        wrapper.__dict__.update(func.__dict__)
        return wrapper
    return decorator
###############################################


In [ ]:
################# GRAPHIC UTILITIES ###################
MOVING_WINDOW = 50
def plot_mean_stds(collections, collection_names, title, xlabel='', ylabel='', ylimits: list[float] = [], ylog=False):
    """
    Plots the means of all data collections on a single graph for direct comparison.
    
    Args:
        collections: List of lists (e.g., [run1_data, run2_data, ...]) for each collection.
        collection_names: Names corresponding to each collection for the legend.
        title: Title of the combined graph.
        xlabel: Label for the x-axis
        ylabel: Label for the y-axis
        ylimits: Optional list specifying the y-axis limits (default: []).
        ylog: Boolean indicating whether to use a logarithmic scale for the y-axis (default: False).
    """
    n = len(collections)
        
    plt.figure(figsize=(10, 6))

    # DEFAULT GRAPHIC SETTINGS FOR BETTER VISUALIZATION
    plt.rcParams.update({
        'font.size': 14,
        'axes.labelsize': 16,
        'axes.titlesize': 18,
        'legend.fontsize': 14,
        'xtick.labelsize': 14,
        'ytick.labelsize': 14,
        'lines.linewidth': 3,           
        'grid.linewidth': 1.5,         
    })
    
    if n == 2:
        colors = ['#0072B2', '#D55E00']  # Scenario 1
    elif n == 4:
        colors = ['#1f77b4', '#aec7e8', '#ff7f0e', '#ffbb78']  # Scenario 2
    elif n == 3:
        colors = ['#0072B2', '#D55E00', "#5b466f"]  # Scenario 3
    else:
        cmap = plt.get_cmap('tab10')
        colors = cmap(np.linspace(0, 1, n))
    
    for i, (data, label) in enumerate(zip(collections, collection_names)):
        smoothed_runs = np.array(data)
        media = np.mean(smoothed_runs, axis=0)
        std = np.std(smoothed_runs, axis=0)
        
        # Average plot
        plt.plot(media, label=f'{label} (Media)', color=colors[i], linewidth=4)
        
        # STD plot (transparent)
        plt.fill_between(
            range(len(media)),
            media - std,
            media + std,
            color=colors[i],
            alpha=0.15
        )
    
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(loc='upper right')
    plt.ylim(ylimits if ylimits else None)
    if ylog:
        plt.yscale('log')
    plt.tight_layout()
    plt.show()
def moving_average(dati, window=10000):
    """Computes the moving average to smooth out noise in the plot."""
    return [np.mean(dati[max(0, i - window + 1):i + 1]) for i in range(len(dati))]


In [ ]:
################### EXPERIMENT UTILITIES ###################

@repeat_and_average(n_runs=5)
def run_experiment(
    learning_agent: DQNPlayer, 
    TOTAL_PLAYERS: int, 
    MAX_PLAYS: int, 
    TOTAL_CARDS: int, 
    JACKS: int) -> tuple[OrderedDict[int, list[int]], OrderedDict[int, list[list[bitarray]]]]:
    """Runs the experiment for a specified number of episodes, tracking the cumulative rewards and actions of each player.
    Args:
        game (Game): game environment.
        players (list[Player]): list of players involved in the experiment, including the learning agent.
        learning_agent (DQNPlayer): learning agent to be monitored.
        MAX_PLAYS (int): The maximum number of episodes to run.
        TOTAL_CARDS (int): The total number of cards in the game.
        JACKS (int): The number of jokers in the game.
    Returns:
        tuple[OrderedDict[Player, list[int]], OrderedDict[Player, list[list[bitarray]]]]: A tuple containing the cumulative rewards for each player and the history of actions for each round.
    """
    deck = Deck(num_cards=TOTAL_CARDS, jacks=JACKS)  # Exclude jokers for the deck
    game = Game(deck=deck)
    players: list[Player] = [RandomPlayer(player_id=i, total_cards=TOTAL_CARDS - JACKS) for i in range(TOTAL_PLAYERS - 1)]  # Create 3 random players
    players.append(learning_agent)  # Add the learning agent to the list of players
    rewards_for_player: OrderedDict[int, list[int]] = OrderedDict((int(player.get_id()), []) for player in players)  # Initialize cumulative rewards for each player
    for player in players:
        game.add_player(player)  # Ensure all players are added to the game environment

    actions_history: OrderedDict[int, list[list[bitarray]]] = OrderedDict((int(player.get_id()), []) for player in players)  
    # For each player, for each round in an episode, we will store the action taken as a bitarray in a list. 
    for episode in range(MAX_PLAYS):
        print(f"\n\nEpisode {episode + 1}/{MAX_PLAYS}")
        game.reset()  # Reset the game environment for a new episode
        game.start_game()  # Start the game

        round = 0
        episode = 0
        total_cards_played = bitarray(TOTAL_CARDS)  # Initialize a bitarray to keep track of all played cards in the current episode
        
        while True:
            print(f"Round {episode + 1}")
            [print(f"Player {player.get_id()} hand cards left:\t{sum(player._hand)}") for player in game.players]  # Debug: Print each player's hand before playing

            actions = run_actions(game, players, learning_agent)  # Get actions for all players
            for player in players:
                if len(actions_history[int(player.get_id())]) <= round:
                    actions_history[int(player.get_id())].append([])  # Initialize the list for this round if it doesn't exist
                actions_history[int(player.get_id())][round].append(actions[player])  # Store the action taken by each player in the history

            game_results, _, done, truncated, _ = game.step(actions)
            
            print(f"Played cards: {game_results['played_cards']}")  # Debug: Print the results of the game step
            print(f"total_cards_played before update: {total_cards_played}")  # Debug: Print the total cards played before updating
            total_cards_played |= game_results["played_cards"]  # Update the total cards played
            print_game_state(game, actions, total_cards_played, game_results["rewards"])

            update_players_state(players, actions, game_results)  # Update the state of each player based on the actions and results


            [rewards_for_player[int(player.get_id())].append(game_results["rewards"][player]) for player in players]  # Track cumulative rewards for each player

            if done or truncated:
                print(f"Game Over\n")
                round = 0
                break
            round += 1
            episode += 1
    return rewards_for_player, actions_history  # Return the cumulative rewards for the learning agent after all episodes


def run_actions(game: Game, players: list[Player], learning_agent: DQNPlayer) -> OrderedDict[Player, bitarray]:
    """
    Executes the actions of the players and returns an ordered dictionary with the actions.
    """

    actions = OrderedDict()
    played_cards = 0  # Counter of cards played in this round before the current turn

    for player in players:
        if player is learning_agent:
            learning_agent.num_cards_played = played_cards
            
        action = player.play_cards()
        actions[player] = action
    
        played_cards += action.count()

    return actions


def update_players_state(players: list[Player], actions: OrderedDict[Player, bitarray], game_results: dict):
    """
    Aggiorna lo stato di ogni giocatore in base alle azioni e ai risultati del gioco.
    """
    for player in players:
        player.update_state(
            actions[player],
            game_results["played_cards"], 
            game_results["rewards"][player]
        )






In [ ]:
############################# 2 x 4 EXPERIMENT SETTING #############################
TOTAL_PLAYERS = 2
CARDS_PER_PLAYER = 4
JACKS = 0
TOTAL_CARDS = TOTAL_PLAYERS * CARDS_PER_PLAYER  # Total cards including jokers

In [ ]:
# ---------------- FULL DECK SPACE AGENT --------------------- #
FULL_DECK_AGENT_2x4 = DQNPlayer(
    player_id=TOTAL_PLAYERS - 1,  # Assign the last player ID to the learning agent
    total_cards=TOTAL_CARDS, 
    cards_per_player=TOTAL_CARDS // TOTAL_PLAYERS,  # Calculate cards per player based on total cards and players
    discount_factor=0.99,
    batch_size=64,
    SYNC_TARGET_EVERY=400,
    replay_buffer_capacity=MAX_PLAYS
)

FULL_DECK_rewards_for_player_2x4, FULL_DECK_actions_history_2x4 = run_experiment(
    FULL_DECK_AGENT_2x4,
    TOTAL_PLAYERS, MAX_PLAYS, TOTAL_CARDS, JACKS=JACKS
)
# ----------------------------------------------------------- #

In [ ]:
# ---------------- COMPRESSED SPACE AGENT --------------------- #
COMPRESSED_AGENT_2x4 = DQNPlayer_HandBitmap(
    player_id=TOTAL_PLAYERS - 1,  # Assign the last player ID to the learning agent
    total_cards=TOTAL_CARDS, 
    cards_per_player=TOTAL_CARDS // TOTAL_PLAYERS,  # Calculate cards per player based on total cards and players
    discount_factor=0.99,
    batch_size=64,
    SYNC_TARGET_EVERY=400,
    replay_buffer_capacity=MAX_PLAYS
)

COMPRESSED_rewards_for_player_2x4, COMPRESSED_actions_history_2x4 = run_experiment(
    COMPRESSED_AGENT_2x4,
    TOTAL_PLAYERS, MAX_PLAYS, TOTAL_CARDS, JACKS=JACKS
)
# ----------------------------------------------------------- #

In [ ]:
############################# 3 x 4 EXPERIMENT SETTING #############################
TOTAL_PLAYERS = 3
CARDS_PER_PLAYER = 4
TOTAL_CARDS = TOTAL_PLAYERS * CARDS_PER_PLAYER  # Total cards including jokers
JACKS = 0

In [ ]:
# ---------------- FULL DECK SPACE AGENT --------------------- #
FULL_DECK_AGENT_3x4 = DQNPlayer(
    player_id=TOTAL_PLAYERS - 1,  # Assign the last player ID to the learning agent
    total_cards=TOTAL_CARDS, 
    cards_per_player=TOTAL_CARDS // TOTAL_PLAYERS,  # Calculate cards per player based on total cards and players
    discount_factor=0.99,
    batch_size=64,
    SYNC_TARGET_EVERY=400,
    replay_buffer_capacity=MAX_PLAYS
)

FULL_DECK_rewards_for_player_3x4, FULL_DECK_actions_history_3x4 = run_experiment(
    FULL_DECK_AGENT_3x4,
    TOTAL_PLAYERS, MAX_PLAYS, TOTAL_CARDS, JACKS=JACKS
)

# ----------------------------------------------------------- #

In [ ]:
# ---------------- COMPRESSED SPACE AGENT --------------------- #
COMPRESSED_AGENT_3x4 = DQNPlayer_HandBitmap(
    player_id=TOTAL_PLAYERS - 1,  # Assign the last player ID to the learning agent
    total_cards=TOTAL_CARDS, 
    cards_per_player=TOTAL_CARDS // TOTAL_PLAYERS,  # Calculate cards per player based on total cards and players
    discount_factor=0.99,
    batch_size=64,
    SYNC_TARGET_EVERY=400,
    replay_buffer_capacity=MAX_PLAYS
)

COMPRESSED_rewards_for_player_3x4, COMPRESSED_actions_history_3x4 = run_experiment(
    COMPRESSED_AGENT_3x4,
    TOTAL_PLAYERS, MAX_PLAYS, TOTAL_CARDS, JACKS=JACKS
)
# ----------------------------------------------------------- #

In [ ]:
############# PLOT ESTIMATION ERRORS #############
MOVING_WINDOW = 50
FULL_DECK_estimation_errors_2x4, FULL_DECK_target_errors_2x4 = FULL_DECK_AGENT_2x4.get_errors()  # Get the estimation and target errors from the learning agent
FULL_DECK_estimation_errors_3x4, FULL_DECK_target_errors_3x4 = FULL_DECK_AGENT_3x4.get_errors()  # Get the estimation and target errors from the learning agent
COMPRESSED_estimation_errors_2x4, COMPRESSED_target_errors_2x4 = COMPRESSED_AGENT_2x4.get_errors()  # Get the estimation and target errors from the learning agent
COMPRESSED_estimation_errors_3x4, COMPRESSED_target_errors_3x4 = COMPRESSED_AGENT_3x4.get_errors()  # Get the estimation and target errors from the learning agent
plot_mean_stds(
    [
        [moving_average(FULL_DECK_target_errors_2x4, window=MOVING_WINDOW)], 
        [moving_average(COMPRESSED_target_errors_2x4, window=MOVING_WINDOW)], 
    ], 
    collection_names=["Full Deck Target Values", "Compressed Target Values",],
    title="Target Values Comparison (2x4)",
    xlabel="Episodes",
    ylabel="Average Target Value",
)  # Passiamo una lista di run, anche se ne abbiamo solo una
plot_mean_stds(
    [
        [moving_average(FULL_DECK_target_errors_3x4, window=MOVING_WINDOW)],
        [moving_average(COMPRESSED_target_errors_3x4, window=MOVING_WINDOW)],
    ], 
    collection_names=["Full Deck Target Values", "Compressed Target Values"],
    title="Target Values Comparison (3x4)",
    xlabel="Episodes",
    ylabel="Average Target Value",
)  # Passiamo una lista di run, anche se ne abbiamo solo una



In [ ]:
MOVING_WINDOW = 1000
FULL_DECK_difference_2x4 = [FULL_DECK_estimation_errors_2x4[i] - FULL_DECK_target_errors_2x4[i] for i in range(len(FULL_DECK_target_errors_2x4))]  # Create a list of indices for the x-axis
COMPRESSED_difference_2x4 = [COMPRESSED_estimation_errors_2x4[i] - COMPRESSED_target_errors_2x4[i] for i in range(len(COMPRESSED_target_errors_2x4))]  # Create a list of indices for the x-axis
FULL_DECK_difference_3x4 = [FULL_DECK_estimation_errors_3x4[i] - FULL_DECK_target_errors_3x4[i] for i in range(len(FULL_DECK_target_errors_3x4))]  # Create a list of indices for the x-axis
COMPRESSED_difference_3x4 = [COMPRESSED_estimation_errors_3x4[i] - COMPRESSED_target_errors_3x4[i] for i in range(len(COMPRESSED_target_errors_3x4))]  # Create a list of indices for the x-axis
plot_mean_stds(
    [
        [moving_average(FULL_DECK_difference_2x4, window=MOVING_WINDOW)], 
        [moving_average(COMPRESSED_difference_2x4, window=MOVING_WINDOW)], ], 
    collection_names=["Full Deck error", "Compressed error"],
    title="TD Errors Comparison (2x4)",
    xlabel="Episodes",
    ylabel="Average Error per Episode",
) 
plot_mean_stds(
    [
        [moving_average(FULL_DECK_difference_3x4, window=MOVING_WINDOW)], 
        [moving_average(COMPRESSED_difference_3x4, window=MOVING_WINDOW)], ], 
    collection_names=["Full Deck error", "Compressed error"],
    title="TD Errors Comparison (3x4)",
    xlabel="Episodes",
    ylabel="Average Error per Episode",
) 

In [ ]:
################ PLOT REWARDS ################
MOVING_WINDOW = 100 
plot_mean_stds(
    [
        [moving_average(FULL_DECK_rewards_for_player_2x4[FULL_DECK_AGENT_2x4.get_id()], window=MOVING_WINDOW)], 
        [moving_average(COMPRESSED_rewards_for_player_2x4[COMPRESSED_AGENT_2x4.get_id()], window=MOVING_WINDOW)], 
        [moving_average(COMPRESSED_rewards_for_player_2x4[0], window=MOVING_WINDOW)],
    ], 
    collection_names=["Full Deck Rewards", "Compressed Rewards","Random Player Rewards"],
    title="Rewards Comparison (2x4)",
    xlabel="Episodes",
    ylabel="Average Rewards per Episode",
) 
plot_mean_stds(
    [
        [moving_average(FULL_DECK_rewards_for_player_3x4[FULL_DECK_AGENT_3x4.get_id()], window=MOVING_WINDOW)],
        [moving_average(COMPRESSED_rewards_for_player_3x4[COMPRESSED_AGENT_3x4.get_id()], window=MOVING_WINDOW)],
        [moving_average(COMPRESSED_rewards_for_player_3x4[1], window=MOVING_WINDOW)],
    ], 
    collection_names=["Full Deck Rewards", "Compressed Rewards","Random Player Rewards"],
    title="Rewards Comparison (3x4)",
    xlabel="Episodes",
    ylabel="Average Rewards per Episode",
)  